# Анализ Gating Weights (α) — DGCA Fusion на RESD

Исследуем, как модель распределяет доверие между текстовой и аудио модальностями:
- Глобальное распределение α
- Примеры с высоким / низким α_text
- Зависимость α от класса эмоции
- Какие размерности стабильно предпочитают одну модальность
- Верные vs неверные предсказания: есть ли разница в α?

## 1. Install & clone

In [ ]:
import subprocess, sys, os
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'datasets', 'librosa', 'scikit-learn', 'tqdm', 'sentencepiece',
], check=True)

REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Done.')

## 2. Config & imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoFeatureExtractor,
    AutoProcessor, AutoModelForSpeechSeq2Seq,
)
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from tqdm.auto import tqdm
import librosa
warnings.filterwarnings('ignore')

# ── пути к моделям (Kaggle input) ─────────────────────────────────────────────
FUSION_PT  = '/kaggle/input/models/aleksandribryanov/fusion-resd/pytorch/default/1/best_dgca_resd.pt'
BERT_CKPT  = '/kaggle/input/models/aleksandribryanov/fusion-resd/pytorch/default/1'  # директория

WHISPER_MODEL = 'artyomboyko/whisper-small-ru-v2'
BERT_MODEL    = 'Aniemore/rubert-tiny2-russian-emotion-detection'
WAVLM_MODEL   = 'Aniemore/wavlm-emotion-russian-resd'

SR_TARGET    = 16_000
MAX_TEXT_LEN = 128
MAX_AUDIO_S  = 10.0
NUM_HEADS    = 4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

RESD_LABEL2ID = {
    'happiness': 0, 'sadness': 1, 'anger': 2,
    'fear': 3, 'disgust': 4, 'enthusiasm': 5, 'neutral': 6,
}
RESD_LABELS = ['happiness', 'sadness', 'anger', 'fear', 'disgust', 'enthusiasm', 'neutral']
NUM_CLASSES = 7
OUT_DIR = Path('/kaggle/working')

## 3. Загрузка моделей

In [ ]:
# config.json может называться 'config (1).json'
import shutil
for wrong, right in [('config (1).json', 'config.json')]:
    p = Path(BERT_CKPT) / wrong
    if p.exists() and not (Path(BERT_CKPT) / right).exists():
        shutil.copy(p, Path(BERT_CKPT) / right)
        print(f'Renamed {wrong} → {right}')

print('Loading BERT...')
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_CKPT)
bert_backbone  = AutoModel.from_pretrained(BERT_CKPT).to(device)
bert_backbone.eval()
BERT_DIM = bert_backbone.config.hidden_size

print('Loading WavLM...')
wavlm_processor = AutoFeatureExtractor.from_pretrained(WAVLM_MODEL)
wavlm_backbone  = AutoModel.from_pretrained(WAVLM_MODEL).to(device)
wavlm_backbone.eval()
WAVLM_DIM = wavlm_backbone.config.hidden_size

print(f'BERT_DIM={BERT_DIM}  WAVLM_DIM={WAVLM_DIM}')

In [ ]:
class DGCAFusion(nn.Module):
    def __init__(self, d_text, d_audio, D, num_heads, num_classes, dropout=0.0):
        super().__init__()
        self.proj_text  = nn.Linear(d_text,  D)
        self.proj_audio = nn.Linear(d_audio, D)
        self.mha_t2a    = nn.MultiheadAttention(D, num_heads, batch_first=True, dropout=dropout)
        self.mha_a2t    = nn.MultiheadAttention(D, num_heads, batch_first=True, dropout=dropout)
        self.ln_text    = nn.LayerNorm(D)
        self.ln_audio   = nn.LayerNorm(D)
        self.gate_text  = nn.Linear(D, D)
        self.gate_audio = nn.Linear(D, D)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(D, num_classes)

    def forward_with_alpha(self, h_text, h_audio):
        """Возвращает (logits, alpha_text, alpha_audio) — все (B, D)."""
        T = self.proj_text(h_text).unsqueeze(1)
        A = self.proj_audio(h_audio).unsqueeze(1)
        T_ref = self.ln_text(T  + self.mha_t2a(T, A, A)[0]).squeeze(1)
        A_ref = self.ln_audio(A + self.mha_a2t(A, T, T)[0]).squeeze(1)
        G_t = self.gate_text(T_ref)
        G_a = self.gate_audio(A_ref)
        gates   = F.softmax(torch.stack([G_t, G_a], dim=-1), dim=-1)
        alpha_t = gates[..., 0]  # (B, D)
        alpha_a = gates[..., 1]
        fused   = alpha_t * T_ref + alpha_a * A_ref
        return self.classifier(self.dropout(fused)), alpha_t, alpha_a


print(f'Loading fusion from {FUSION_PT}')
ckpt = torch.load(FUSION_PT, map_location=device, weights_only=False)
FUSION_DIM = ckpt['fusion']['proj_text.weight'].shape[0]
print(f'  step={ckpt["step"]}  val_wacc={ckpt["val_wacc"]:.4f}  D={FUSION_DIM}')

fusion = DGCAFusion(BERT_DIM, WAVLM_DIM, FUSION_DIM, NUM_HEADS, NUM_CLASSES).to(device)
fusion.load_state_dict(ckpt['fusion'])
fusion.eval()
print(f'Fusion loaded. Params: {sum(p.numel() for p in fusion.parameters()):,}')

## 4. Транскрипция RESD test

In [ ]:
print('Loading Whisper...')
asr_proc  = AutoProcessor.from_pretrained(WHISPER_MODEL)
asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)
asr_model.eval()

@torch.no_grad()
def transcribe(wav_np):
    dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
    inp    = asr_proc(wav_np, sampling_rate=SR_TARGET, return_tensors='pt')
    ids    = asr_model.generate(inp.input_features.to(device, dtype=dtype), language='ru', task='transcribe')
    return asr_proc.batch_decode(ids, skip_special_tokens=True)[0]

ds_test = load_dataset('Aniemore/resd', split='test')
print(f'Test samples: {len(ds_test)}')

records = []
for ex in tqdm(ds_test, desc='Transcribing'):
    wav = np.array(ex['speech']['array'], dtype=np.float32)
    sr  = ex['speech']['sampling_rate']
    if sr != SR_TARGET:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
    records.append({
        'text':   transcribe(wav),
        'audio':  wav,
        'label':  RESD_LABEL2ID[ex['emotion']],
        'emotion': ex['emotion'],
    })

del asr_model, asr_proc
torch.cuda.empty_cache()
print('Done.')

## 5. Вычисление α для всех примеров

In [ ]:
MAX_AUDIO_LEN = int(MAX_AUDIO_S * SR_TARGET)

@torch.no_grad()
def encode_text(texts):
    enc = bert_tokenizer(texts, truncation=True, padding='max_length',
                         max_length=MAX_TEXT_LEN, return_tensors='pt')
    out = bert_backbone(enc['input_ids'].to(device), enc['attention_mask'].to(device))
    return out.last_hidden_state[:, 0, :]

@torch.no_grad()
def encode_audio_single(wav_np):
    inp    = wavlm_processor(wav_np, sampling_rate=SR_TARGET, return_tensors='pt')
    hidden = wavlm_backbone(inp['input_values'].to(device)).last_hidden_state
    return hidden.mean(dim=1)  # (1, D)


results = []  # list of dicts per sample

BATCH = 8
for i in tqdm(range(0, len(records), BATCH), desc='Computing α'):
    batch = records[i:i + BATCH]
    texts  = [r['text']  for r in batch]
    labels = [r['label'] for r in batch]
    emotions = [r['emotion'] for r in batch]

    h_text = encode_text(texts)  # (B, BERT_DIM)
    h_audio = torch.cat([encode_audio_single(r['audio'][:MAX_AUDIO_LEN]) for r in batch], dim=0)

    logits, alpha_t, alpha_a = fusion.forward_with_alpha(h_text, h_audio)
    probs = F.softmax(logits, dim=-1)

    for j in range(len(batch)):
        pred = logits[j].argmax().item()
        conf = probs[j].max().item()
        at   = alpha_t[j].cpu().numpy()  # (D,)
        results.append({
            'text':       texts[j],
            'emotion':    emotions[j],
            'label':      labels[j],
            'pred':       pred,
            'correct':    pred == labels[j],
            'confidence': conf,
            'alpha_text_mean': float(at.mean()),
            'alpha_text_vec':  at,
        })

df = pd.DataFrame([{k: v for k, v in r.items() if k != 'alpha_text_vec'} for r in results])
alpha_matrix = np.stack([r['alpha_text_vec'] for r in results])  # (N, D)
print(f'Done. Shape: {alpha_matrix.shape}')
print(df[['emotion', 'correct', 'alpha_text_mean', 'confidence']].describe())

## 6. Глобальное распределение α

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# гистограмма всех значений α_text по всем сэмплам и размерностям
axes[0].hist(alpha_matrix.flatten(), bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(0.5, color='red', linestyle='--', label='α=0.5')
axes[0].set_xlabel('α_text (d)')
axes[0].set_ylabel('Count')
axes[0].set_title('Распределение α_text по всем сэмплам и размерностям')
axes[0].legend()

# распределение среднего α_text на сэмпл
axes[1].hist(df['alpha_text_mean'], bins=40, color='darkorange', edgecolor='white', linewidth=0.3)
axes[1].axvline(df['alpha_text_mean'].mean(), color='red', linestyle='--',
                label=f'mean={df["alpha_text_mean"].mean():.3f}')
axes[1].set_xlabel('mean(α_text) per sample')
axes[1].set_ylabel('Count')
axes[1].set_title('Среднее α_text на сэмпл')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'alpha_global.png', dpi=150)
plt.show()
print(f'Глобальное mean(α_text)={alpha_matrix.mean():.3f}  mean(α_audio)={1-alpha_matrix.mean():.3f}')

## 7. Примеры с высоким и низким α_text

In [ ]:
df_sorted = df.sort_values('alpha_text_mean')

print('=== Аудио доминирует (α_text низкое) ===')
for _, row in df_sorted.head(5).iterrows():
    mark = '✓' if row['correct'] else '✗'
    print(f'  [{mark}] α_text={row["alpha_text_mean"]:.3f}  '
          f'true={row["emotion"]:12s}  pred={RESD_LABELS[int(row["pred"]):int(row["pred"])+1][0]:12s}  '
          f'text="{row["text"][:60]}"')

print()
print('=== Текст доминирует (α_text высокое) ===')
for _, row in df_sorted.tail(5).iterrows():
    mark = '✓' if row['correct'] else '✗'
    print(f'  [{mark}] α_text={row["alpha_text_mean"]:.3f}  '
          f'true={row["emotion"]:12s}  pred={RESD_LABELS[int(row["pred"]):int(row["pred"])+1][0]:12s}  '
          f'text="{row["text"][:60]}"')

## 8. α по классам эмоций

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = df.groupby('emotion')['alpha_text_mean'].median().sort_values().index.tolist()
sns.boxplot(data=df, x='emotion', y='alpha_text_mean', order=order,
            palette='coolwarm', ax=ax)
ax.axhline(0.5, color='black', linestyle='--', alpha=0.5, label='α=0.5 (равный вклад)')
ax.set_xlabel('Эмоция')
ax.set_ylabel('mean(α_text) per sample')
ax.set_title('α_text по классам эмоций')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'alpha_by_class.png', dpi=150)
plt.show()

print('Медиана α_text по классам:')
print(df.groupby('emotion')['alpha_text_mean'].agg(['median', 'mean', 'std']).round(3).to_string())

## 9. Верные vs неверные предсказания

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# α_text: верные vs неверные
for correct, label, color in [(True, 'Верные', 'green'), (False, 'Неверные', 'red')]:
    vals = df[df['correct'] == correct]['alpha_text_mean']
    axes[0].hist(vals, bins=30, alpha=0.6, label=f'{label} (n={len(vals)})', color=color)
axes[0].set_xlabel('mean(α_text)')
axes[0].set_title('α_text: верные vs неверные предсказания')
axes[0].legend()

# уверенность модели vs α_text
scatter = axes[1].scatter(
    df['alpha_text_mean'], df['confidence'],
    c=df['correct'].map({True: 'green', False: 'red'}),
    alpha=0.5, s=20,
)
axes[1].set_xlabel('mean(α_text)')
axes[1].set_ylabel('Confidence (max softmax)')
axes[1].set_title('Уверенность vs α_text')
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color='green', label='Верно'), Patch(color='red', label='Неверно')])

plt.tight_layout()
plt.savefig(OUT_DIR / 'alpha_correct_vs_wrong.png', dpi=150)
plt.show()

print('mean(α_text) по группам:')
print(df.groupby('correct')['alpha_text_mean'].agg(['mean', 'median', 'std']).round(3))

## 10. Стабильно «текстовые» и «аудио» размерности

In [ ]:
# среднее α_text по каждой из D размерностей по всем сэмплам
dim_mean = alpha_matrix.mean(axis=0)  # (D,)
dim_std  = alpha_matrix.std(axis=0)

idx_sorted = np.argsort(dim_mean)
top_audio = idx_sorted[:10]   # размерности, где аудио доминирует
top_text  = idx_sorted[-10:]  # размерности, где текст доминирует

print(f'Топ-10 «аудио» размерностей (низкий α_text): {top_audio}')
print(f'  mean α_text: {dim_mean[top_audio].round(3)}')
print(f'Топ-10 «текст» размерностей (высокий α_text): {top_text}')
print(f'  mean α_text: {dim_mean[top_text].round(3)}')

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(dim_mean[idx_sorted], color='steelblue')
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5)
axes[0].fill_between(range(len(dim_mean)),
                     dim_mean[idx_sorted] - dim_std[idx_sorted],
                     dim_mean[idx_sorted] + dim_std[idx_sorted],
                     alpha=0.2, color='steelblue')
axes[0].set_xlabel('Размерность (отсортировано по α_text)')
axes[0].set_ylabel('mean α_text')
axes[0].set_title('Среднее α_text по каждой размерности (±std)')

# heatmap: α_text по классам и размерностям (топ-20 «текст» + топ-20 «аудио»)
top40 = np.concatenate([idx_sorted[:20], idx_sorted[-20:]])
per_class_alpha = np.array([
    alpha_matrix[df['label'].values == cls, :][:, top40].mean(axis=0)
    for cls in range(NUM_CLASSES)
])  # (7, 40)

sns.heatmap(per_class_alpha, ax=axes[1],
            xticklabels=[f'd{i}' for i in top40],
            yticklabels=RESD_LABELS,
            cmap='RdYlGn', vmin=0, vmax=1,
            cbar_kws={'label': 'α_text'})
axes[1].set_title('α_text по классам × топ-40 размерностей (20 «аудио» + 20 «текст»)')
axes[1].set_xlabel('Размерность (левые — аудио, правые — текст)')

plt.tight_layout()
plt.savefig(OUT_DIR / 'alpha_dimensions.png', dpi=150)
plt.show()

## 11. Итоговая сводка

In [ ]:
acc  = accuracy_score(df['label'], df['pred'])
wacc = balanced_accuracy_score(df['label'], df['pred'])

print('=== Summary ===')
print(f'Accuracy         : {acc:.4f}')
print(f'Weighted Accuracy : {wacc:.4f}')
print()
print(f'Global mean α_text  : {alpha_matrix.mean():.3f}  (>0.5 → текст доминирует в среднем)')
print(f'Global mean α_audio : {1 - alpha_matrix.mean():.3f}')
print()
print(f'Доля размерностей с α_text > 0.5 : {(dim_mean > 0.5).mean():.1%}')
print(f'Доля размерностей с α_text < 0.5 : {(dim_mean < 0.5).mean():.1%}')
print()
corr = np.corrcoef(df['alpha_text_mean'], df['confidence'])[0, 1]
print(f'Корреляция (α_text, confidence) : {corr:.3f}')
print()
print('mean(α_text) верные / неверные:')
print(df.groupby('correct')['alpha_text_mean'].mean().round(3))